<a href="https://colab.research.google.com/github/takatakamanbou/AdvML/blob/2025/AdvML2025_ex12notebookA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AdvML ex12notebookA

<img width=72 src="https://www-tlab.math.ryukoku.ac.jp/~takataka/course/AdvML/AdvML-logo.png"> [この授業のウェブページ](https://www-tlab.math.ryukoku.ac.jp/wiki/?AdvML)




板書や口頭で補足する前提なので，この notebook だけでは説明が不完全です．


---
## 次元削減（前回の続き）
---



---
### 非線形な次元削減



前回，代表的な次元削減手法である「主成分分析（PCA）」を紹介した．PCAは非常によく使われる手法であるが，あくまで「線形」な方法であるため，データが非線形な構造を持っている場合には，それをうまくとらえることができない．例えば，データが曲線や曲面のような多様体上に分布しているような場合，PCAで得られる低次元表現は，その構造を潰してしまうことがある．

こうした背景から，より柔軟に非線形構造をとらえることを目的とした非線形次元削減手法が多数提案されている．ここではそれらの中から，Isomap および t-SNE という二つの手法を取り上げる．

- **Isomap**: Isometric Mapping の略．非線形次元削減の古典的手法の一つ．データ点間のユークリッド距離に代えて，ある地形に沿って移動したときの道のり（測地線距離）を近さとして扱い，それを保つようにデータを低次元に埋め込む．
- **t-SNE**: t-distributed Stochastic Neighbor Embedding の略．主に高次元データの可視化に使われる．局所的なデータ点間の近さを保ったまま，それらを2次元や3次元に配置する．

Python の機械学習ライブラリ scikit-learn では，[sklearn.manifold.Isomap](https://scikit-learn.org/stable/modules/generated/sklearn.manifold.Isomap.html) および [sklearn.manifold.TSNE](https://scikit-learn.org/stable/modules/generated/sklearn.manifold.TSNE.html) として実装されている．以下では，二種類のデータにこの二手法と主成分分析（PCA）を適用し，2次元に次元削減する実験を行う．PCA は [sklearn.decomposition.PCA](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html) を利用する．

In [ ]:
import numpy as np

import matplotlib.pyplot as plt
from matplotlib import ticker
from matplotlib import offsetbox

from sklearn import datasets
from sklearn.decomposition import PCA
from sklearn.manifold import Isomap, TSNE
from sklearn.preprocessing import MinMaxScaler

---
### 実験1: 3次元データ

この実験は，https://scikit-learn.org/stable/auto_examples/manifold/plot_compare_methods.html を参考にしている．

In [ ]:
# データの作成
n_samples = 1500
S_points, S_color = datasets.make_s_curve(n_samples, random_state=0)
S_points[:, 1] *= 1.5

# データの可視化
x, y, z = S_points.T

fig, ax = plt.subplots(
    figsize=(6, 6),
    facecolor="white",
    tight_layout=True,
    subplot_kw={"projection": "3d"},
)
col = ax.scatter(x, y, z, c=S_color, s=20, alpha=0.8)
ax.view_init(azim=-60, elev=9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(1))
ax.yaxis.set_major_locator(ticker.MultipleLocator(1))
ax.zaxis.set_major_locator(ticker.MultipleLocator(1))
ax.set_xlim(-1, 3)
ax.set_ylim(-2, 2)
ax.set_zlim(-2, 2)

fig.colorbar(col, ax=ax, orientation="horizontal", shrink=0.6, aspect=60, pad=0.01)
plt.show()

上図のように，データが3次元空間中でS字状になった曲面（2次元多様体）上に分布している．色は，結果の可視化を分かりやすくするために付けたものである．PCA, Isomap, t-SNE でこのデータを2次元に次元削減すると，次の結果が得られる．Isomap, t-SNE ともに削減後の次元数以外にもハイパーパラメータがあり，それらを代えると結果が変わるが，ここではその詳細は省略する．

In [ ]:
# 実験条件の設定
n_neighbors = 12  # neighborhood which is used to recover the locally linear structure
n_components = 2  # number of coordinates for the manifold
experiment1 = {
    'PCA': PCA(n_components=n_components),
    'Isomap': Isomap(n_neighbors=n_neighbors, n_components=n_components, p=1),
    't-SNE': TSNE(
        n_components=n_components,
        perplexity=30,
        init="random",
        max_iter=250,
        random_state=0,
    ),
}

# 実験
fig, ax = plt.subplots(1, 3, figsize=(10, 3), facecolor="white", constrained_layout=True)
for i, (name, transformer) in enumerate(experiment1.items()):
    points = transformer.fit_transform(S_points)
    x, y = points.T
    ax[i].scatter(x, y, c=S_color, s=20, alpha=0.8)
    ax[i].set_title(name)
    ax[i].xaxis.set_major_formatter(ticker.NullFormatter())
    ax[i].yaxis.set_major_formatter(ticker.NullFormatter())
plt.show()

PCAはデータを平面に射影するだけで曲面の構造をとらえられていないが，Isomap と t-SNE その曲面を平面にほどけていることが分かる．

---
### 実験2: 64次元の手書き数字画像

この実験は，https://scikit-learn.org/stable/auto_examples/manifold/plot_lle_digits.html を参考にしている．

In [ ]:
# データの読み込み
digits = datasets.load_digits(n_class=6)
X, y = digits.data, digits.target
n_samples, n_features = X.shape
print(f'X.shape = {X.shape}')

# データの可視化
fig, axs = plt.subplots(nrows=10, ncols=10, figsize=(4, 4))
for idx, ax in enumerate(axs.ravel()):
    ax.imshow(X[idx].reshape((8, 8)), cmap=plt.cm.binary)
    ax.axis("off")
plt.show()

データは，グレイスケールの手書き数字画像（0から5までの6種類）である．縦横8画素なので，データの次元数は64である．このデータに PCA, Isomap, t-SNE を適用して2次元に次元削減してみよう．

In [ ]:
# 実験結果の可視化のための関数の定義

def plot_embedding(X, title):
    _, ax = plt.subplots()
    X = MinMaxScaler().fit_transform(X)

    for digit in digits.target_names:
        ax.scatter(
            *X[y == digit].T,
            marker=f"${digit}$",
            s=60,
            color=plt.cm.Dark2(digit),
            alpha=0.425,
            zorder=2,
        )
    shown_images = np.array([[1.0, 1.0]])  # just something big
    for i in range(X.shape[0]):
        # plot every digit on the embedding
        # show an annotation box for a group of digits
        dist = np.sum((X[i] - shown_images) ** 2, 1)
        if np.min(dist) < 4e-3:
            # don't show points that are too close
            continue
        shown_images = np.concatenate([shown_images, [X[i]]], axis=0)
        imagebox = offsetbox.AnnotationBbox(
            offsetbox.OffsetImage(digits.images[i], cmap=plt.cm.gray_r), X[i],
        )
        imagebox.set(zorder=1)
        ax.add_artist(imagebox)

    ax.set_title(title)
    ax.axis("off")

In [ ]:
# 実験条件の設定
n_components = 2
n_neighbors = 30

experiment2 = {
    "PCA": PCA(n_components=2),
    "Isomap": Isomap(n_neighbors=n_neighbors, n_components=n_components),
    "t-SNE": TSNE(
        n_components=n_components,
        max_iter=500,
        n_iter_without_progress=150,
        n_jobs=2,
        random_state=0,
    ),
}

# 実験
for name, transformer in experiment2.items():
    Y = transformer.fit_transform(X, y)
    plot_embedding(Y, name)
plt.show()

PCAでは6クラスの数字が入り交じっているが，Isomap や t-SNE ではクラスがある程度分離された結果が得られることが分かる．

---
## オートエンコーダ
---

---
### オートエンコーダとは

板書もしながら説明します．

- Encoder（符号化器）: $D$次元の値 $\pmb{x}$ を入力すると $H < D$ 次元の値 $\pmb{z}$ を出力する変換．$\pmb{z} = \textrm{Enc}(\pmb{x})$
- Decoder（復号器）: $H < D$ 次元の値 $\pmb{z}$ を入力すると $D$ 次元の値 $\hat{\pmb{x}}$ を出力する変換．$\hat{\pmb{x}} = \textrm{Dec}(\pmb{z})$

Enc, Dec をニューラルネットで構成し，$\hat{\pmb{x}} = \textrm{Dec}(\textrm{Enc}(\pmb{x}))$ と $\pmb{x}$ との間の誤差を最小化するように学習させると，$D$ 次元のデータを $H$ 次元にしてから $D$ 次元で再構成する仕組みを作れる．これを **オートエンコーダ** (**Autoencoder**) という．

学習データを

$$
\{ \pmb{x}_n \in \mathbb{R}^{D} | n = 1, 2, \ldots, N\}
$$

とするとき，学習のための損失関数としては，学習データ自身とその再構成との間の二乗誤差を用いるのが典型的．

$$
L = \sum_{n=1}^{N} \Vert \pmb{x}_n - \textrm{Dec}(\textrm{Enc}(\pmb{x}_n))\Vert^2
$$



---
### 線形オートエンコーダ

オートエンコーダをニューラルネットで作る最も単純な方法は，$\textrm{Dec}(\textrm{Enc}(\pmb{x}))$ を，中間層を一つだけもつ 2 層の階層型ニューラルネットワークで構成する方法である．

この場合，
中間層のニューロン数を $H < D$，出力層のニューロン数を $D$ として，上記の $L$ を損失関数として学習を行う．
出力層ニューロンの活性化関数は恒等関数とする（$\sigma(x) = x$とする）ことが多い（注）．

<br>
<hr width="50%" align="left">
<span style="font-size: 75%">
※ 注: 値の範囲が限られるデータを扱う場合，再構成される値が範囲からはみ出さないように，入力を $[0, 1]$ や $[-1, 1]$ に規格化したうえで，出力層ニューロンの活性化関数にロジスティックシグモイドや $\tanh$ を用いることもある．
</span>


このような2層ニューラルネットで $L$ を損失関数とする場合，学習によって得られる $L$ の値は，主成分分析で次元数を $H$ として次元削減→再構成した場合に得られる二乗誤差を下回らないことが知られている．
このことは，中間層のニューロンに非線形の活性化関数を用いても変わらない．

中間層ニューロンがバイアス項を持たず活性化関数が恒等関数である場合，この層には $H \times D$ 個のパラメータがある．それらの値を並べた $H\times D$ 行列を $A$ とおくと， $\textrm{Enc}(\pmb{x}) = A\pmb{x}$ と表せる．
また，出力層ニューロンもバイアス項を持たず活性化関数が恒等関数である場合，この層の $D \times H$ 個のパラメータを並べた $D \times H$ 行列を $B$ とおくと，$\textrm{Dec}(\pmb{z}) = B\pmb{z}$ と表せる．
中間層と出力層がこのように線形変換である場合，オートエンコーダ全体の入出力は $\textrm{Dec}(\textrm{Enc}(\pmb{x})) = BA\pmb{x}$ となる．
以下，このようなオートエンコーダを線形オートエンコーダと呼ぶことにする．

主成分分析の「再構成とその誤差」の節で述べたように，データの分散共分散行列の固有値の大きい方から $H$ 個に対応する固有ベクトルをならべた $D\times H$ 行列を $U_H$ としたとき， 任意の $H\times H$ 正則行列 $C$ に対して $A = CU_H^{\top}, B = U_H C^{-1}$ とおけば，再構成の二乗誤差が最小となる．
平均 $\pmb{0}$ のデータに対して $L$ を損失関数として学習させた線形オートエンコーダのパラメータ $A, B$ は，この形に収束することが知られている．



---
### 実験: 手書き数字画像の次元削減

手書数字画像のデータセット MNIST のサブセットに主成分分析および線形オートエンコーダを適用して，次元削減して再構成する実験を行ってみよう．

#### 準備

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn
seaborn.set_theme()

# scikit-learn のいろいろ
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

# PyTorch 関係のほげ
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import ToTensor
import torchsummary

if torch.cuda.is_available():
    device = 'cuda'
else:
    device = 'cpu'
print(device)

In [ ]:
# MNIST データセットの入手
Xraw, yraw = fetch_openml('mnist_784', version=1, parser='auto', return_X_y=True, as_frame=False)
Xall = Xraw[:20000] / 255.0     # 画素値が [0, 255] の整数値なので [0, 1] の浮動小数点数値に変換
yall = yraw[:20000].astype(int) # クラスラベル．0 から 9 の整数値（この実験では使わない）

# 学習データとテストデータの分割
XL, XT, yL, yT = train_test_split(Xall, yall, test_size=4000, random_state=4649, stratify=yall)
NL, NT = len(XL), len(XT)
D = XL.shape[1]

# 学習データの平均を求め，平均を引いたデータを作る
Xm = np.mean(XL, axis=0)
XL2 = XL - Xm

print(f'XL.shape = {XL.shape}')
print(f'XT.shape = {XT.shape}')
print(f'Xm.shape = {Xm.shape}')

In [ ]:
# 学習データの最初の50枚を可視化
nrow, ncol = 5, 10
fig, ax = plt.subplots(nrow, ncol, figsize=(0.6*ncol, 0.6*nrow))
for i in range(nrow):
    for j in range(ncol):
        img = XL[i*ncol + j, ::].reshape((28, 28))
        ax[i, j].imshow(img, cmap=plt.cm.gray, vmin=0, vmax=1)
        ax[i, j].axis('off')

fig.tight_layout()
plt.show()

#### 主成分分析の適用

主成分分析では，学習データの分散共分散行列の固有値と固有ベクトルを求める必要があります．その手順は，素直に考えれば，(1) 学習データの分散共分散行列を求め，(2) その固有値と固有ベクトルを計算する，という二段階です．しかし，「特異値分解」(Singular Value Decomposition, SVD)という行列分解を利用すると，平均を $\pmb{0}$ にした学習データの行列に SVD を適用するだけで，求める固有値・固有ベクトルが得られます（詳細の説明は省略します）．次のコードセルでは，そのようにしています．

In [ ]:
# 平均を引いたデータ行列をSVD
_, sva, Vt = np.linalg.svd(XL2, full_matrices=False)
eva = sva**2/len(XL)  # 固有値
U = Vt                # 固有ベクトルをならべた行列
print(f'eva.shape = {eva.shape}')
print(f'U.shape = {U.shape}')

# 固有値の大きい方 10 個の値
print(eva[:10])

In [ ]:
# 固有値から累積寄与率を算出
cumeva = np.cumsum(eva)
cumeva /= cumeva[-1]

# グラフ
fig = plt.figure(figsize=(8, 6))
# 固有値
ax1 = fig.add_subplot(221)
ax1.plot(np.arange(len(eva))+1, eva, label='eigenvalues')
ax1.axhline(0, color='gray')
ax1.legend()
# 固有値（次元数の範囲を限定）
ax2 = fig.add_subplot(222)
ax2.plot(np.arange(len(eva))+1, eva, '.', label='eigenvalues')
ax2.axhline(0, color='gray')
ax2.set_xlim(-5, 100)
ax2.legend()
# 累積寄与率
ax3 = fig.add_subplot(223)
ax3.plot(np.arange(len(eva))+1, cumeva, label='cumulative contribution rate')
ax3.axhline(1, color='gray')
ax3.axhline(0.9, color='gray', linestyle='--')
ax3.axhline(0.8, color='gray', linestyle='--')
ax3.legend()
# 累積寄与率（次元数の範囲を限定）
ax4 = fig.add_subplot(224)
ax4.plot(np.arange(len(eva))+1, cumeva, '.', label='cumulative contribution rate')
ax4.axhline(1, color='gray')
ax4.axhline(0.9, color='gray', linestyle='--')
ax4.axhline(0.8, color='gray', linestyle='--')
ax4.set_xlim(-5, 100)
ax4.legend()
fig.tight_layout()
plt.show()

上段はデータの分散共分散行列の固有値，下段はそこから算出される累積寄与率を示す．
いずれの図においても，横軸は次元数を表す．

固有値を降順に $\lambda_h$ ($h = 1, 2, \ldots, D)$ とおくとき，累積寄与率 $c_h$ は次式で計算される量である．

$$
c_h = \frac{\sum_{d=1}^{h}\lambda_d}{\sum_{d=1}^{D}\lambda_d}
$$

この式の分母は，$y_1$ から $y_D$ までの変数の分散の総和に等しい．さらに，この値は，元のデータの$D$個すべての変数の分散の総和に一致する．分子は，$y_1$ から $y_h$ までの変数の分散の総和である．したがって，$c_h$ は，変換後の $h$ 個の変数だけで元データの分散のうちどの程度の割合を表せるかを示す．

In [ ]:
# 平均と固有ベクトルを可視化
nrow, ncol = 5, 10
fig, ax = plt.subplots(nrow, ncol, figsize=(0.6*ncol, 0.6*nrow))
for i in range(nrow):
    for j in range(ncol):
        if i == 0 and j == 0:
            uu = Xm
        else:
            uu = U[i*ncol +j - 1, ::]
            uumax = np.max(np.abs(uu))
            uu = uu / (uumax*2) + 0.5
        img = uu.reshape((28, 28))
        ax[i, j].imshow(img, cmap=plt.cm.gray, vmin=0, vmax=1)
        ax[i, j].axis('off')

fig.tight_layout()
plt.show()

上図左上の画像は，平均を可視化したものである．それ以外は，固有ベクトルを可視化したものである．左上から右に向かって，対応する固有値の大きい順に並んでいる．固有ベクトルは正負の値をとるので，ここでは値 $0$ がちょうど真っ黒と真っ白の中間の灰色になるように規格化している．

#### 主成分分析による再構成

学習データの分散共分散行列の固有値を $\lambda_1 \geq \lambda_2 \geq \dots \geq\lambda_D$ とおき，これらに対応する単位固有ベクトルを $\pmb{u}_h$（$h = 1, 2, \ldots, D$）とおく．$h = 1$ から $h = H \leq D$ までの固有ベクトルを並べた $D\times H$ 行列を $U_H$ とおく．
このとき，一つのデータ $\pmb{x}$ を $H$ 次元に次元削減してから再構成する計算は，次式で与えられる．

$$
\hat{\pmb{x}} = U_H\pmb{z} + \pmb{\mu} = U_HU_H^{\top}(\pmb{x} -  \pmb{\mu}) + \pmb{\mu}
$$

ただし，$\pmb{\mu}$ は学習データの平均である．

次のコードセルを実行すると，$H$ をいろいろ変えながら学習データを再構成し，元データとの間の二乗誤差を求める．

In [ ]:
### 次元数を変えながら学習データを再構成して二乗誤差を計算

# 学習データを変換
Z = XL2 @ U.T

# 次元数 H の設定
HList = np.hstack((
    [0], np.arange(1, 20), np.arange(20, 50, 5),
    np.arange(50, 400, 10), np.arange(400, 700, 50), [700, D]
))

# 平均二乗誤差の配列
mse = np.zeros(len(HList))

# 0 番目は平均をそのまま再構成とした場合の二乗誤差
mse[0] = np.mean(XL2**2)

for i, H in enumerate(HList[1:]):
    # H次元で再構成
    XX = Z[:, :H] @ U[:H, :] + Xm
    # 元画像と再構成との間の平均二乗誤差
    mse[i] = np.mean((XL - XX)**2)
    print(f'{H} {mse[i]:.6f}')

# 二乗誤差をグラフに描く
fig = plt.figure(figsize=(6, 4))
ax1 = fig.add_subplot(111)
ax1.plot(HList, mse, '.', label='reconstruction error')
ax1.axhline(0, color='gray')
ax1.legend()
plt.show()

#### 線形オートエンコーダの学習

ここでは，PyTorch を用いて線形オートエンコーダを実装し，学習させてみる．

In [ ]:
# データを扱うためのクラス
#
class MMDataset(Dataset):

    def __init__(self, dataX):
        self.X = dataX

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        X = torch.tensor(self.X[idx], dtype=torch.float32)
        return X

In [ ]:
# 線形オートエンコーダ
#
class LinearAE(nn.Module):

    def __init__(self, D, H):
        super(LinearAE, self).__init__()
        # Encoder
        self.enc = nn.Linear(D, H, bias=False)
        # Decoder
        self.dec = nn.Linear(H, D, bias=False)

    def forward(self, X):
        X = self.enc(X)
        X = self.dec(X)
        return X

In [ ]:
# 学習の関数
#
def train(model, lossFunc, optimizer, dl):
    loss_sum = 0.0
    n = 0
    for i, X in enumerate(dl):
        X = X.to(device)
        Xhat = model(X)         # 一つのバッチ X を入力して出力 Xhat を計算
        loss = lossFunc(Xhat, X) # 入力 X を正解として loss を計算
        optimizer.zero_grad()   # 勾配をリセット
        loss.backward()         # 誤差逆伝播でパラメータ更新量を計算
        optimizer.step()         # パラメータを更新
        n += len(X)
        loss_sum += loss.item()  # 損失関数の値

    return loss_sum/n

In [ ]:
# 損失関数の値を求める関数
#
@torch.no_grad()
def evaluate(model, lossFunc, dl):
    loss_sum = 0.0
    n = 0
    for i, X in enumerate(dl):
        X = X.to(device)
        Xhat = model(X)         # 一つのバッチ X を入力して出力 Xhat を計算
        loss = lossFunc(Xhat, X)  # 入力 X を正解として loss を計算
        n += len(X)
        loss_sum += loss.item() # 損失関数の値

    return loss_sum/n

In [ ]:
# データ読み込みの仕組みを作る
dsL = MMDataset(XL - Xm)
dsT = MMDataset(XT - Xm)
dlL = DataLoader(dsL, batch_size=100, shuffle=True)
dlT = DataLoader(dsT, batch_size=100, shuffle=False)

次のコードセルの `H` が，線形オートエンコーダの中間層ニューロン数である．このオートエンコーダが入力データを何次元に次元削減するかを定める．

次のコードセル冒頭の `H` が，線形オートエンコーダの中間層ニューロン数である．このオートエンコーダが入力データを何次元に次元削減するかを定める．

このコードセルを実行すると，線形オートエンコーダを学習させ，エポック毎の損失の値をグラフに描く．

In [ ]:
# ネットワークモデルの定義
H = 100
net = LinearAE(D, H).to(device)

# ネットワークの構造を表示
torchsummary.summary(net, (1, D), device=device)

# 損失関数（二乗誤差） とパラメータ最適化器の設定
loss_func = nn.MSELoss(reduction='sum')
optimizer = torch.optim.Adam(net.parameters(), lr=1e-3)

# 学習の繰り返し回数
nepoch = 30

# 学習
L = []
print(f'学習データ数: {len(dsL)}  テストデータ数: {len(dsT)}')
print()
print('# epoch  lossL  lossT')
for t in range(1, nepoch+1):
    lossL = train(net, loss_func, optimizer, dlL) / D
    lossT = evaluate(net, loss_func, dlT) / D
    L.append([t, lossL, lossT])
    if (t < 10) or (t % 10 == 0):
        print(f'{t}   {lossL:.5f}   {lossT:.5f}')

# 学習曲線の表示
data = np.array(L)
fig = plt.figure()
ax = fig.add_subplot(111)
ax.plot(data[:, 0], data[:, 1], '.-', label='loss for training data')
ax.plot(data[:, 0], data[:, 2], '.-', label='loss for test data')
ax.axhline(0.0, color='gray')
ax.legend()
ax.set_title(f'loss')
plt.show()

# 学習後の損失
loss2 = evaluate(net, loss_func, dlL) / D
print(f'# lossL: {loss2:.5f}', end='   ')
loss2 = evaluate(net, loss_func, dlT) / D
print(f'# lossT: {loss2:.5f}')

`lossL` の値は，「主成分分析による再構成」の実験で求めた二乗誤差と対応している．同じ次元数のときのPCAの再構成誤差の値と比較してみよう．


#### 線形オートエンコーダによる再構成

次のコードセルを実行すると，上で学習させた線形オートエンコーダを用いて，テストデータの一部を再構成したものを可視化する．比較のため，主成分分析で同じ次元数に次元削減してから再構成したものも並べるようにしてある．

In [ ]:
# テストデータの最初の 1 バッチの出力を求める
for i, X in enumerate(dlT):
    X = X.to(device)
    Xhat = net(X)
    break

# 元画像
XX = X.to('cpu').detach().numpy() + Xm

# 線形オートエンコーダによる再構成
XhatAE = Xhat.to('cpu').detach().numpy() + Xm

# PCA による再構成
UH = U[:H, :]
XhatPCA = (XX - Xm) @ UH.T @ UH + Xm

# 再構成したテストデータの最初の10枚を可視化
ncol = 10
fig, ax = plt.subplots(3, ncol, figsize=(0.8*ncol, 0.8*3))

for j in range(ncol):
    # 元画像
    img = XX[j, ::].reshape((28, 28))
    ax[0, j].imshow(img, cmap=plt.cm.gray, vmin=0, vmax=1)
    ax[0, j].axis('off')
    # オートエンコーダで再構成した画像
    img = XhatAE[j, ::].reshape((28, 28))
    ax[1, j].imshow(img, cmap=plt.cm.gray, vmin=0, vmax=1)
    ax[1, j].axis('off')
    # PCAで再構成した画像
    img = XhatPCA[j, ::].reshape((28, 28))
    ax[2, j].imshow(img, cmap=plt.cm.gray, vmin=0, vmax=1)
    ax[2, j].axis('off')

fig.tight_layout()
plt.show()